# 87 Ablations on the GPU (A100)

Each run removes or changes one part of the final system, so the report can show what that part adds. Every comparison uses the same questions as the final system.

| Run | What changes | What it shows |
|---|---|---|
| **B3** | candidates from the SFT generator (before RL), chosen by `judge_v2` | does RL still matter once the verifier chooses? |
| **B5** | `judge_untrained`: the plain base model asked the verifier's question, with the result, and no training | does the verifier need training, or would a prompt do? |
| **B1** | `judge_v2_own`: trained on its generator's own answers only (no base-model pool) | does the extra training data matter? |
| **B4** | `judge_v2_nocost`: trained without the cost loss | does the auxiliary cost loss change correctness? |
| **B2** | `judge_v2_base`: started from the plain base model, not the generator's adapter | does starting from the generator matter? |

Everything else is exactly as for `judge_v2` (notebook 85): the same data, settings and steps. Each judge then scores the same candidates of validation, dev and mini-dev. Nothing here chooses anything: the comparison is made in notebook 88.

**How to run.** *Runtime > Change runtime type > A100*, then *Runtime > Run all*. About **8 hours, roughly 90 to 100 units**. Keep the browser tab open.

**After a disconnect:** reopen this notebook from the GitHub link and press *Run all* again. Every step checks Drive first and skips what is finished. A training run continues from its last checkpoint (at most 50 steps, about 15 minutes, are redone).

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import subprocess


def run(cmd):
    """Run a command, show its output, and stop "Run all" if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"failed (exit {p.returncode}): {cmd}")


import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
DATA = "judge_data_v1"        # the verifier's training data (train questions only)
DATA_OWN = "judge_data_own"   # B1: the same, own answers only
CAND = "judge_cand_v1"        # the final generator's candidates for val, dev and mini-dev
SFT = "sft_v1_best_e1"        # B3: the generator before RL (RL started from it)
CAND_SFT = "judge_cand_sft"   # B3: its candidates
SPLITS = "val,dev,minidev"
for f in (f"{DATA}/exec_train.jsonl", f"{DATA}/exec_val.jsonl", f"{SFT}/adapter/adapter_config.json",
          f"{CAND}/exec_dev.jsonl", "judge_v2/judge_meta.json"):
    assert os.path.exists(f"{R}/{f}"), f"{f} is missing on Drive"
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])
run("nvidia-smi --query-gpu=name,memory.total --format=csv")
print("CPU cores:", os.cpu_count())

### Stage the databases (validation and dev; a minute, skipped when already there)

In [ ]:
run(f"effsql stage --split train --db-ids {VAL_DBS}")
run("effsql stage --split dev")

### B3. Candidates from the SFT generator, run with the 1 s cap, scored by `judge_v2` (about 45 minutes)

In [ ]:
run(f"python -m effsql.judge candidates --run {CAND_SFT} --generator {SFT} --splits {SPLITS}")
run(f"python -m effsql.judge exec --run {CAND_SFT} --splits {SPLITS}")
run(f"python -m effsql.judge score --run {CAND_SFT} --judge judge_v2 --splits {SPLITS}")

### B5. The untrained verifier (about 25 minutes)

In [ ]:
run("python -m effsql.judge untrained --run judge_untrained")
run(f"python -m effsql.judge score --run {CAND} --judge judge_untrained --splits {SPLITS}")

### B1. The verifier trained on its generator's own answers only (about 1.5 hours)

The data is the existing `judge_data_v1` without the base-model candidates (about 63% of the candidates remain); the answer-time results are carried over, so nothing is run again.

In [ ]:
run(f"python -m effsql.judge derive-data --src {DATA} --run {DATA_OWN} --sources own")
run(f"python -m effsql.judge train --run judge_v2_own --data {DATA_OWN} --with-results")
run(f"python -m effsql.judge score --run {CAND} --judge judge_v2_own --splits {SPLITS}")

### B4. The verifier trained without the cost loss (about 2.3 hours)

In [ ]:
run(f"python -m effsql.judge train --run judge_v2_nocost --data {DATA} --with-results --cost-weight 0")
run(f"python -m effsql.judge score --run {CAND} --judge judge_v2_nocost --splits {SPLITS}")

### B2. The verifier started from the plain base model (about 2.3 hours)

In [ ]:
run(f"python -m effsql.judge train --run judge_v2_base --data {DATA} --with-results --init-from none")
run(f"python -m effsql.judge score --run {CAND} --judge judge_v2_base --splits {SPLITS}")

### B3 labels: run the SFT generator's candidates with the evaluation's limits (CPU of this machine, about 25 minutes)

Used only to score the choices in notebook 88, never when answering.

In [ ]:
run(f"python -m effsql.judge label --run {CAND_SFT} --splits {SPLITS}")

### Summary: each verifier on the held-out validation databases

`auc`: how well the score separates correct from wrong candidates (0.5 is chance). `top1`: among questions with a correct candidate, how often the top-scored one is correct. The untrained verifier has no such numbers here (it was never trained); notebook 88 compares all of them on the same candidates.

In [ ]:
for j in ("judge_v2", "judge_v2_own", "judge_v2_nocost", "judge_v2_base"):
    v = json.load(open(f"{R}/{j}/judge_meta.json"))["val_final"]
    print(f"{j:<16}", {k: round(v[k], 3) for k in ("logloss", "accuracy", "auc", "top1")})
for c, j in [(CAND_SFT, "judge_v2")] + [(CAND, j) for j in ("judge_untrained", "judge_v2_own", "judge_v2_nocost", "judge_v2_base")]:
    for s in SPLITS.split(","):
        assert os.path.exists(f"{R}/{c}/scores_{j}_{s}.done.json"), f"{j} on {c} {s} is not finished"
print("all ablation runs are finished")

### Done: disconnect now (*Runtime > Disconnect and delete runtime*), then run `88_ablations_report` on a CPU runtime